# Sports Pick Analysis

Tracks aggregated recommendations and accuracy over time.
Run `main.py fetch` or `main.py add-pick` first to populate the database.

In [ ]:
import sys
sys.path.insert(0, '.')

import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

import db
db.init_db()

sns.set_theme(style='darkgrid')
DB_PATH = Path('data/picks.db')
print('DB path:', DB_PATH.resolve())

## Today's Recommendations

In [ ]:
today = datetime.utcnow().strftime('%Y-%m-%d')
rows = db.load_aggregated_for_date(today)
if rows:
    df = pd.DataFrame(rows)
    display(df[['sport','home_team','away_team','market_type','pick_side','pick_team',
                'composite_score','edge_flag','recommended_stake',
                'sources_agreeing','sources_disagreeing']].reset_index(drop=True))
else:
    print(f'No aggregated picks for {today}. Run: python main.py fetch --sport MLB')

## Source Accuracy Over Time

In [ ]:
accuracy_rows = db.accuracy_by_source()
if accuracy_rows:
    acc = pd.DataFrame(accuracy_rows)
    acc['win_rate'] = acc['wins'] / acc['total']
    display(acc)

    fig, ax = plt.subplots(figsize=(10, 5))
    pivot = acc.pivot(index='source', columns='market_type', values='win_rate')
    pivot.plot.bar(ax=ax, alpha=0.8)
    ax.axhline(0.5, color='red', linestyle='--', alpha=0.5, label='50% breakeven')
    ax.set_title('Win Rate by Source and Market Type')
    ax.set_ylabel('Win Rate')
    ax.set_ylim(0, 1)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No settled outcomes yet. Log results with: python main.py log-outcome ...')

## Edge Flag Performance

Tracks whether the Kalshi-derived edge zones (15-35¢ underdogs, near-even picks) are actually predictive for your sports picks.

In [ ]:
conn = sqlite3.connect('data/picks.db')
conn.row_factory = sqlite3.Row

edge_rows = conn.execute("""
    SELECT
        a.edge_flag,
        COUNT(*) AS total,
        SUM(CASE
            WHEN (a.pick_side IN ('HOME','AWAY') AND a.pick_side = o.winner) THEN 1
            ELSE 0 END) AS wins
    FROM aggregated_picks a
    JOIN games g ON a.game_id = g.id
    LEFT JOIN outcomes o ON g.id = o.game_id
    WHERE o.winner IS NOT NULL
    GROUP BY a.edge_flag
""").fetchall()

if edge_rows:
    edge_df = pd.DataFrame([dict(r) for r in edge_rows])
    edge_df['win_rate'] = edge_df['wins'] / edge_df['total']
    display(edge_df)
else:
    print('No settled picks yet.')

conn.close()

## P&L Tracker

Requires `log-outcome` data. Assumes $1/stake-level unit sizing.

In [ ]:
conn = sqlite3.connect('data/picks.db')
conn.row_factory = sqlite3.Row

pnl_rows = conn.execute("""
    SELECT
        g.game_time,
        a.sport,
        a.edge_flag,
        a.recommended_stake,
        a.pick_side,
        o.winner,
        CASE WHEN a.pick_side = o.winner THEN a.recommended_stake ELSE -a.recommended_stake END AS pnl
    FROM aggregated_picks a
    JOIN games g ON a.game_id = g.id
    JOIN outcomes o ON g.id = o.game_id
    ORDER BY g.game_time
""").fetchall()

if pnl_rows:
    pnl = pd.DataFrame([dict(r) for r in pnl_rows])
    pnl['game_time'] = pd.to_datetime(pnl['game_time'])
    pnl['cumulative_pnl'] = pnl['pnl'].cumsum()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(pnl['game_time'], pnl['cumulative_pnl'], linewidth=2)
    ax.axhline(0, color='red', linestyle='--', alpha=0.5)
    ax.fill_between(pnl['game_time'], pnl['cumulative_pnl'],
        where=pnl['cumulative_pnl'] >= 0, alpha=0.2, color='green')
    ax.fill_between(pnl['game_time'], pnl['cumulative_pnl'],
        where=pnl['cumulative_pnl'] < 0, alpha=0.2, color='red')
    ax.set_title('Cumulative P&L (units)')
    ax.set_ylabel('Units')
    plt.tight_layout()
    plt.show()

    print(f'Total: {pnl["pnl"].sum():.1f} units  |  W/L: {(pnl["pnl"]>0).sum()}-{(pnl["pnl"]<0).sum()}')
else:
    print('No settled picks yet.')

conn.close()